# Wide Expert-Parallelism for DeepSeek-R1

DeepSeek-R1 is a 671-billion parameter Mixture-of-Experts (MoE) model.
Unlike dense models, MoE models activate only a subset of their parameters
("experts") for each token. DeepSeek-R1 uses 256 experts and activates 8
per token, so only ~21B parameters are active at any given time.

**Wide Expert-Parallelism (EP)** distributes experts across multiple nodes,
rather than replicating all experts on every node. This provides two
key advantages:

1. **More KV-cache space**: With experts spread across nodes, each node
   uses less GPU memory for weights, leaving more for KV-cache.
2. **Higher throughput**: More KV-cache means more concurrent requests
   can be served without evictions.

This notebook deploys DeepSeek-R1 using the LeaderWorkerSet (LWS)
pattern for multi-node vLLM, configures the DP/EP topology, and
benchmarks inference performance.

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MODEL_NAME"] = "deepseek-ai/DeepSeek-R1"

print("Namespace:", os.environ["NAMESPACE"])
print("Model:", os.environ["MODEL_NAME"])
print()
print("DeepSeek-R1 specifications:")
print("  Total parameters:     671B")
print("  Active parameters:    ~21B per token (8 of 256 experts)")
print("  Architecture:         MoE (Mixture of Experts)")
print("  Expert count:         256 routed experts + shared experts")
print("  Experts per token:    8")

## MoE Architecture and Why Wide EP Helps

### Standard Tensor Parallelism (TP)

With standard TP, each GPU holds a slice of every layer (including all
experts). For DeepSeek-R1 at FP8, the model weights alone need ~670 GB.
On 8x H100 (80 GB each = 640 GB total), there is almost no room left
for KV-cache.

### Expert Parallelism (EP)

With EP, experts are partitioned across GPUs. Each GPU holds only a
fraction of the experts. When a token needs an expert on a different GPU,
the activations are sent over the interconnect (NVLink or InfiniBand).

### Wide EP (Multi-Node)

Wide EP extends this across nodes. Instead of 8 GPUs on 1 node, you
use 8 GPUs across 2 or 4 nodes. Each node holds fewer experts, freeing
up massive amounts of GPU memory for KV-cache:

```
Standard TP (1 node, 8 GPUs):
  Each GPU: all experts (shard) + small KV-cache
  Total KV-cache: ~80 GB across all GPUs

Wide EP (4 nodes, 8 GPUs each = 32 GPUs):
  Each GPU: 1/4 of experts + large KV-cache
  Total KV-cache: ~1.2 TB across all GPUs
```

The tradeoff is inter-node communication latency for expert routing,
but with InfiniBand interconnect, this is manageable for decode-heavy
workloads.

In [ ]:
# Deploy DeepSeek-R1 using the wide EP guide with LeaderWorkerSet
# LeaderWorkerSet (LWS) is a Kubernetes pattern for multi-node workloads
# where one pod is the "leader" and others are "workers"

# First, install the LeaderWorkerSet CRD if not already present
!kubectl apply --server-side -f https://github.com/kubernetes-sigs/lws/releases/latest/download/manifests.yaml

print("LeaderWorkerSet controller installed.")
print("Waiting for controller to be ready...")
!kubectl wait --for=condition=ready pod -l control-plane=controller-manager \
    -n lws-system --timeout=120s
print("LWS controller is ready.")

In [ ]:
# Deploy the model server with wide EP topology
# The guide configures a LeaderWorkerSet with the correct DP/EP parameters

!kubectl apply -k llm-d-production-stack/guides/wide-ep-lws/model-server/ \
    -n $NAMESPACE

print("DeepSeek-R1 model server submitted.")
print()
print("This deployment creates a LeaderWorkerSet with:")
print("  Replicas:           1 group (leader + workers)")
print("  GPUs per worker:    8 (one full node)")
print("  Workers per group:  4 nodes")
print("  Total GPUs:         32")
print()
print("DP/EP topology:")
print("  Data Parallelism (DP):    1 (single data-parallel instance)")
print("  Expert Parallelism (EP):  32 (experts spread across 32 GPUs)")
print("  Tensor Parallelism (TP):  1 (no intra-expert tensor splitting)")
print()
print("Waiting for all pods to be ready (model download may take 20+ minutes)...")
!kubectl wait --for=condition=ready pod -l app=deepseek-r1 -n $NAMESPACE --timeout=1800s
print("All model server pods are ready.")

In [ ]:
# Check the LeaderWorkerSet status

print("=== LeaderWorkerSet Status ===")
!kubectl get leaderworkerset -n $NAMESPACE

print()
print("=== Pods ===")
!kubectl get pods -n $NAMESPACE -l app=deepseek-r1 -o wide

print()
print("=== Node Placement ===")
!kubectl get pods -n $NAMESPACE -l app=deepseek-r1 \
    -o custom-columns='POD:.metadata.name,NODE:.spec.nodeName,ROLE:.metadata.labels.role,GPUS:.spec.containers[0].resources.limits.nvidia\.com/gpu'

In [ ]:
# Deploy the router configured for the LWS topology
# The router needs to know about the multi-node layout to send
# requests to the leader pod, which coordinates across workers

!kubectl apply -k llm-d-production-stack/guides/wide-ep-lws/router/ \
    -n $NAMESPACE

print("Router deployed for wide EP topology.")
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s
print("EPP pods are ready.")

print()
print("The router sends requests to the leader pod of each LWS group.")
print("The leader coordinates with worker pods for expert routing.")
print("From the client perspective, this is transparent.")

In [ ]:
# Send a test request to DeepSeek-R1
import subprocess
import json

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Gateway: {GATEWAY_IP}:8080")
print()
print("Sending a test request to DeepSeek-R1...")

result = subprocess.run(
    ["curl", "-s", f"http://{GATEWAY_IP}:8080/v1/chat/completions",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "deepseek-ai/DeepSeek-R1",
         "messages": [{"role": "user", "content": "What is 25 * 37? Think step by step."}],
         "max_tokens": 300,
     })],
    capture_output=True, text=True
)

response = json.loads(result.stdout)
if "choices" in response:
    content = response["choices"][0]["message"]["content"]
    usage = response.get("usage", {})
    print("Response:")
    print(content[:500])
    print()
    print(f"Prompt tokens:     {usage.get('prompt_tokens', 'N/A')}")
    print(f"Completion tokens: {usage.get('completion_tokens', 'N/A')}")
else:
    print("Response:")
    print(json.dumps(response, indent=2))

In [ ]:
# Run inference benchmarks
# Measure throughput in tokens per second per GPU

import time
import threading

stats = {"tokens": 0, "requests": 0}
stop_event = threading.Event()

def benchmark_worker():
    while not stop_event.is_set():
        payload = json.dumps({
            "model": "deepseek-ai/DeepSeek-R1",
            "messages": [{"role": "user",
                          "content": "Write a short paragraph about quantum computing."}],
            "max_tokens": 200,
        })
        result = subprocess.run(
            ["curl", "-s", "-m", "60",
             f"http://{GATEWAY_IP}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json", "-d", payload],
            capture_output=True, text=True
        )
        try:
            resp = json.loads(result.stdout)
            tokens = resp.get("usage", {}).get("completion_tokens", 0)
            stats["tokens"] += tokens
            stats["requests"] += 1
        except (json.JSONDecodeError, KeyError):
            pass

# Run 16 concurrent benchmark threads for 2 minutes
print("Running throughput benchmark (16 concurrent requests, 2 minutes)...")
print()

threads = [threading.Thread(target=benchmark_worker, daemon=True)
           for _ in range(16)]
start_time = time.time()
for t in threads:
    t.start()

for i in range(8):
    time.sleep(15)
    elapsed = time.time() - start_time
    tps = stats["tokens"] / elapsed if elapsed > 0 else 0
    tps_per_gpu = tps / 32  # 32 GPUs total
    print(f"  [{elapsed:.0f}s] Requests: {stats['requests']:>4}, "
          f"Tokens: {stats['tokens']:>6}, "
          f"Throughput: {tps:.0f} tok/s total, "
          f"{tps_per_gpu:.0f} tok/s per GPU")

stop_event.set()
elapsed = time.time() - start_time
final_tps = stats["tokens"] / elapsed
final_tps_per_gpu = final_tps / 32

print()
print(f"Final throughput: {final_tps:.0f} tok/s total")
print(f"Per-GPU throughput: {final_tps_per_gpu:.0f} tok/s")
print()
print("Reference benchmarks (B200 GPUs, decode phase):")
print("  ~3.1k tok/s per GPU on decode with wide EP")
print("  This scales nearly linearly with GPU count.")

In [ ]:
# Monitor expert utilization across nodes
# With wide EP, we can see which experts are most frequently activated

print("=== Expert Utilization Across Nodes ===")
print()

# Query expert routing metrics
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=vllm_expert_forward_count{namespace="llm-d"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
results = data.get('data', {}).get('result', [])
if results:
    # Group by node
    by_node = {}
    for r in results:
        node = r['metric'].get('node', 'unknown')
        count = int(float(r['value'][1]))
        by_node.setdefault(node, []).append(count)
    for node, counts in sorted(by_node.items()):
        total = sum(counts)
        avg = total / len(counts)
        print(f'  {node}: {len(counts)} experts, {total} total forwards, avg {avg:.0f} per expert')
else:
    print('  (expert metrics not yet available)')
" 2>/dev/null || echo "  (metrics endpoint not reachable)"

print()
print("Inter-node communication stats:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=vllm_expert_comm_latency_seconds{namespace="llm-d",quantile="0.99"}' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    val = float(r['value'][1]) * 1000
    print(f'  Inter-node expert routing p99 latency: {val:.1f} ms')
" 2>/dev/null || echo "  (communication metrics not yet available)"

print()
print("Healthy wide EP deployment characteristics:")
print("  - Expert utilization should be roughly even across nodes")
print("  - Inter-node latency should be <1ms with InfiniBand")
print("  - KV-cache utilization should be significantly higher than")
print("    standard TP (more memory available per GPU for caching)")

## Summary

Wide Expert-Parallelism enables efficient serving of large MoE models
like DeepSeek-R1 by distributing experts across multiple nodes.

### Key Benefits

- **More KV-cache per GPU**: Fewer experts per GPU means more memory for
  caching, supporting higher concurrency.
- **Linear throughput scaling**: Adding nodes adds both compute and cache
  capacity.
- **~3.1k tok/s per GPU**: Reference decode throughput on B200 GPUs.

### Deployment Details

- Uses **LeaderWorkerSet** for multi-node coordination.
- Deploy using `guides/wide-ep-lws` in the production stack.
- The router sends requests to the leader pod; expert routing is
  handled internally by vLLM.

### Hardware Requirements

- Multiple GPU nodes (4 nodes with 8 GPUs each for DeepSeek-R1)
- High-bandwidth interconnect (InfiniBand recommended for inter-node
  expert routing)
- LeaderWorkerSet CRD installed in the cluster

### Next Steps

- **Tiered Prefix Cache**: Combine wide EP with CPU/disk cache offloading
  for even more KV-cache capacity.
- **Autoscaling**: Configure the WVA to manage multiple model deployments
  including wide EP groups.
- **P/D Disaggregation**: Use separate prefill and decode pools, each
  with their own EP topology optimized for their workload phase.